# System Prompt Engineering

This notebook experiments with **improved system prompts** for `Qwen3-4B-Thinking-2507`.

Problems diagnosed from baseline outputs:
- **Overthinking easy questions**: trivial problems receive thousands of thinking tokens.
- **Confidence spiraling on MCQs**: model correctly computes the answer but enters an infinite re-verification loop when the result does not match an option exactly.
- **Redundant verification**: same calculation re-derived multiple times, inflating runtime.

Changes made here:
1. **MCQ prompt**: adds difficulty calibration, anti-spiral commit instruction, and conciseness rules.
2. **Free-form prompt**: adds one-pass instruction — compute once, state result, no re-verification.
3. **`/no_think` mode**: Qwen3 supports a soft prompt suffix to suppress extended thinking; used for MCQs to cut runtime.
4. **Per-type token caps**: MCQ capped at 2048 new tokens, free-form at 8192.

The public dataset (`data/public.jsonl`) contains questions **with** answers for local evaluation.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

True
12.1
1
NVIDIA A30


In [2]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [3]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "1"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We define two **improved** system prompts based on failure-mode diagnosis of baseline outputs:

- **MCQ** (`SYSTEM_PROMPT_MCQ`): instructs the model to assess difficulty, commit to the closest option, and never spiral when the exact result is not present in the choices.
- **Free-form** (`SYSTEM_PROMPT_MATH`): instructs one-pass reasoning — no re-verification of steps already computed.

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

**`/no_think` mode**: Qwen3 supports appending `/no_think` to the user message to suppress extended chain-of-thought. This is useful for MCQs (fast, direct answer) or trivially easy free-form questions. Set `no_think=True` in `build_prompt()` to enable it.

In [ ]:
# ── Improved System Prompts ───────────────────────────────────────────────────
#
# Key changes from baseline:
#   MCQ:       adds difficulty assessment, anti-spiral commit instruction,
#              conciseness rule, and closest-match fallback.
#   Free-form: adds one-pass rule — compute each step once, no re-verification.

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician solving a timed exam. "
    "Assess the difficulty first: if the problem is straightforward, solve it directly "
    "without lengthy verification. "
    "For multiple choice: compute the answer, match it to the closest option, and commit. "
    "Do NOT second-guess yourself or re-derive if your computation is clean. "
    "If your exact result does not appear among the options, pick the closest match "
    "or check whether you misread the problem — do NOT spiral into repeated re-derivations. "
    "Be concise. Avoid repeating calculations you have already done. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician solving a timed exam. "
    "Assess difficulty first: trivial problems need only a few lines of reasoning, "
    "not a full derivation. "
    "Solve step by step but do NOT re-verify steps you are already confident about. "
    "Do each calculation once. If the arithmetic is straightforward, state the result directly. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

# Per-type generation token caps.
# MCQ: 2048 — enough for one clean reasoning pass + letter answer.
# Free-form: 8192 — allows multi-step graduate-level problems without excess.
MAX_TOKENS_MCQ  = 2048
MAX_TOKENS_FREE = 8192


def build_prompt(
    question: str,
    options: Optional[list],
    no_think: bool = False,
) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question.

    Args:
        question: The problem statement.
        options:  List of MCQ answer choices, or None for free-form.
        no_think: If True, appends '/no_think' to the user message to suppress
                  Qwen3's extended chain-of-thought thinking mode.
                  Useful for MCQs or trivially easy problems to cut runtime.
    """
    suffix = "\n/no_think" if no_think else ""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}{suffix}"
    return SYSTEM_PROMPT_MATH, question + suffix


# ── Sanity-check with one sample of each type ────────────────────────────────
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} system prompt ──")
    print(sys_p)
    print(f"\n── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

print(f"MCQ  token cap : {MAX_TOKENS_MCQ}")
print(f"Free token cap : {MAX_TOKENS_FREE}")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     enable_prefix_caching=False,
#     gpu_memory_utilization=0.50,
#     max_model_len=8192,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
#     enforce_eager=True, 
#     task="generate",
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [9]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [ ]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Tokenize (padded batch)
# print(f"Generating responses for {len(prompts)} questions...")
# inputs = tokenizer(
#     prompts,
#     return_tensors="pt",
#     padding=True,
#     truncation=True,
#     max_length=16384,
# ).to(llm.device)

# # Generate
# with torch.no_grad():
#     output_ids = llm.generate(
#         **inputs,
#         max_new_tokens=MAX_TOKENS,
#         temperature=0.6,
#         top_p=0.95,
#         top_k=20,
#         repetition_penalty=1.0,
#         do_sample=True,
#     )

# # Decode only the new tokens (strip the prompt)
# responses = []
# for i, out in enumerate(output_ids):
#     new_tokens = out[inputs["input_ids"].shape[1]:]
#     responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

In [12]:
from transformers import TextStreamer

# ── Test batch: first N MCQ + first N free-form (FRQ) in dataset order ─────────
# `data` order is preserved from public.jsonl — these are the first N rows of
# each type as they appear in the file (not necessarily id 0,1,2,...).

N_SAMPLES = 5

NO_THINK_MCQ = True   # suppress extended thinking for MCQs
NO_THINK_FREE = False  # keep thinking enabled for free-form

mcq_batch = [d for d in data if d.get("options")][:N_SAMPLES]
frq_batch = [d for d in data if not d.get("options")][:N_SAMPLES]

# MCQ first, then free-form (swap order here if you prefer)
test_items = [(d, "MCQ", NO_THINK_MCQ) for d in mcq_batch] + [(d, "Free-form", NO_THINK_FREE) for d in frq_batch]

print(f"Running {len(mcq_batch)} MCQ + {len(frq_batch)} free-form = {len(test_items)} generations total.")

responses = []

for item, label, no_think in test_items:
    is_mcq = bool(item.get("options"))
    max_new = MAX_TOKENS_MCQ if is_mcq else MAX_TOKENS_FREE

    system, user = build_prompt(item["question"], item.get("options"), no_think=no_think)

    prompt_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    print(f"\n{'='*60}")
    print(f"  {label}  |  id={item.get('id')}  |  max_new_tokens={max_new}  |  no_think={no_think}")
    print(f"{'='*60}\n")

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=16384,
    ).to(llm.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            do_sample=True,
            streamer=streamer,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    responses.append({"item": item, "label": label, "response": response})

    n_out = len(new_tokens)
    print(f"\n── Finished | output tokens: {n_out} / {max_new} ──")



── Generating Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, positive even whole numbers start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So this is an arithmetic sequence where the first term is 2, the common difference is 2, and we need the sum of the first 325 terms.

Wait, let me recall the formula for the sum of an arithmetic sequence. The sum S of the first n terms of an arithmetic sequence is given by S = n/2 * [2a + (n - 1)d], where a is the first term, d is the common difference. Alternatively, it's also S = n*(a1 + an)/2, where an is the nth term.

Let me confirm: the first positive even whole number is 2, so a1 = 2. The second is 4, so a2 = ื่, so d = 2. The nth term of this sequence is an = a1 + (n - 1)d = 2 + (n - 1)*2 = 2n. Wait, let's check: for n=1, 2*1=2, which is correct. n=2, 2*2=4, correct. n=3, 6, yes, so the nth term is 2n. That's a good check.

So the 325th te

In [ ]:
import re

def extract_boxed(text: str) -> str:
    """Extract the content of the last \\boxed{} in a response."""
    matches = re.findall(r"\\boxed\{([^}]+)\}", text)
    return matches[-1].strip() if matches else "(no boxed answer found)"

print("\n" + "="*60)
print("  QUICK ANSWER CHECK")
print("="*60)

for r in responses:
    item     = r["item"]
    label    = r["label"]
    response = r["response"]
    gold     = item.get("answer", "?")
    predicted = extract_boxed(response)

    print(f"\n{label}  |  id={item.get('id')}")
    print(f"  Gold     : {gold}")
    print(f"  Predicted: {predicted}")

In [13]:
# import time, json, os

# tokenizer.padding_side = "left"
# tokenizer.pad_token = tokenizer.eos_token

# batch_size = 12
# responses = []

# start = time.time()

# for i in range(0, len(data), batch_size):
#     batch = data[i:i + batch_size]

#     print(f"Processing {i} to {i + len(batch)} / {len(data)}")

#     prompts = []

#     for item in batch:
#         system, user = build_prompt(item["question"], item.get("options"))

#         prompt_text = tokenizer.apply_chat_template(
#             [{"role": "system", "content": system},
#              {"role": "user", "content": user}],
#             tokenize=False,
#             add_generation_prompt=True,
#         )

#         prompts.append(prompt_text)

#     inputs = tokenizer(
#         prompts,
#         return_tensors="pt",
#         padding=True,
#         truncation=True,
#         max_length=16384,
#     ).to(llm.device)

#     with torch.no_grad():
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=32768,
#             temperature=0.6,
#             top_p=0.95,
#             do_sample=True,
#         )

#     for j, out in enumerate(output_ids):
#         input_len = inputs["input_ids"].shape[1]
#         new_tokens = out[input_len:]
#         response = tokenizer.decode(new_tokens, skip_special_tokens=True)
#         responses.append(response)

#     elapsed = time.time() - start
#     done = i + len(batch)
#     print(f"Done {done}/{len(data)} | elapsed: {elapsed/60:.1f} min")

# os.makedirs("results", exist_ok=True)

# with open("results/baseline_responses.json", "w") as f:
#     json.dump(responses, f, indent=2)

# print(f"\nSaved {len(responses)} responses to results/baseline_responses.json")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [14]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

# Score only the items we just generated (`responses` is a list of dicts from the cell above).
results = []
for r in tqdm(responses, desc="Scoring"):
    item     = r["item"]
    response = r["response"]
    is_mcq   = bool(item.get("options"))
    gold     = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 1/1126 [00:00<02:18,  8.10it/s]

Scoring complete. 1 results.


## 8. Summary

Print accuracy broken down by question type.

In [15]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    0 /    0  (0.00%)
  Free-form  :    1 /    1  (100.00%)
  Overall    :    1 /    1  (100.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [16]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 1 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!